# RK1828 YOLO26-pose 人体姿态估计 Demo

Ultralytics **YOLO26-pose**：检测人体并回归 **17 个关键点**（鼻/眼/耳/肩/肘/腕/髋/膝/踝）。
端到端无 NMS 设计（host 端仍做一次轻量 NMS 去重），9 个输出头：

| 输出头 | 形状（stride 8/16/32 各一组） | 含义 |
|---|---|---|
| `box_*` | `[1,4,80,80]` `[1,4,40,40]` `[1,4,20,20]` | 框回归 (l,t,r,b)，无 DFL |
| `score_*` | `[1,1,…]` | 人体置信度（单类别 person） |
| `pos_*` | `[1,51,…]` | 17 关键点 × 3（x, y, 置信度 logit） |

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型（可选 HF / HF-Mirror 源） |
| 2 | 选择模型和图片 |
| 3 | 前处理（letterbox 等比缩放 + 灰边填充） |
| 4 | RKNN 推理 |
| 5 | 后处理（框 + 关键点解码 + NMS） |
| 6 | 骨架可视化 |

> 规格说明：**n/s 有 i8 和 fp16 两版；m 只有 i8**——官方 SDK 包里的
> m-pose fp16 权重文件损坏（实为 .rknn 的副本），未收录。
> 参考成绩：yolo26m-pose i8 约 158ms/帧，示例图人体置信度 0.93。

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。


本 demo 下载 YOLO26-pose n/s/m 模型（m 仅 i8）和示例图 little_boy.jpg。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_i8.rknn", "models/yolo26_pose/yolo26n-pose_i8.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_i8.weight", "models/yolo26_pose/yolo26n-pose_i8.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_fp16.rknn", "models/yolo26_pose/yolo26n-pose_fp16.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_fp16.weight", "models/yolo26_pose/yolo26n-pose_fp16.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_i8.rknn", "models/yolo26_pose/yolo26s-pose_i8.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_i8.weight", "models/yolo26_pose/yolo26s-pose_i8.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_fp16.rknn", "models/yolo26_pose/yolo26s-pose_fp16.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_fp16.weight", "models/yolo26_pose/yolo26s-pose_fp16.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26m-pose_i8.rknn", "models/yolo26_pose/yolo26m-pose_i8.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26m-pose_i8.weight", "models/yolo26_pose/yolo26m-pose_i8.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/little_boy.jpg", "datasets/little_boy.jpg"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 选择模型和图片

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_model = widgets.Dropdown(
    options=[("yolo26n-pose i8", "yolo26n-pose_i8"),
             ("yolo26n-pose fp16", "yolo26n-pose_fp16"),
             ("yolo26s-pose i8", "yolo26s-pose_i8"),
             ("yolo26s-pose fp16", "yolo26s-pose_fp16"),
             ("yolo26m-pose i8（仅此版）", "yolo26m-pose_i8")],
    value="yolo26m-pose_i8", description="模型:")
w_image = widgets.Dropdown(options=sorted(glob.glob("datasets/*.jpg")),
                           description="图片:", layout=widgets.Layout(width="60%"))
if any(o.endswith("/little_boy.jpg") for o in w_image.options):
    w_image.value = [o for o in w_image.options if o.endswith("/little_boy.jpg")][0]
display(w_model, w_image)
print("选好后继续往下运行 ▼")

## 3. 前处理：letterbox

YOLO 要求 640×640 正方形输入。直接拉伸会改变人体比例，所以**等比缩放后灰边填充**，
并记下缩放比例和 padding —— 后处理时要用它们把框和关键点还原回原图坐标。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from vision_post import letterbox, to_input_array, setup_cjk_font

setup_cjk_font()
img = Image.open(w_image.value).convert("RGB")
img_lb, SCALE, PAD_X, PAD_Y = letterbox(img, 640)
x = to_input_array(img_lb)
print(f"原图 {img.size} -> 输入 {x.shape}, scale={SCALE:.3f}, pad=({PAD_X},{PAD_Y})")

# 前处理每个阶段都画出来：原图 -> letterbox -> 输入张量
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, im, title in [
    (axes[0], img, f"1. 原图 {img.size[0]}x{img.size[1]}"),
    (axes[1], img_lb, "2. letterbox 640x640\n(等比缩放 + 灰边填充)"),
    (axes[2], x[0], f"3. 模型输入 {x.shape}\nuint8 NHWC"),
]:
    ax.imshow(np.asarray(im))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. RKNN 推理

In [ ]:
import time
from rknn3 import Rknn3Model

model = Rknn3Model(f"models/yolo26_pose/{w_model.value}.rknn")
print(f"{len(model.output_attrs)} 个输出头:")
for a in model.output_attrs:
    print("  ", a)

_ = model.infer([x])
t0 = time.time()
outputs = model.infer([x])
print(f"推理耗时: {(time.time()-t0)*1000:.1f} ms")

## 5. 后处理：框 + 关键点解码

- 每个网格：box 头给出 (l,t,r,b) 距离，score 头给出人体置信度
- 关键点：`x = (raw_x + 网格x + 0.5) × stride`，置信度再过 sigmoid
- 全部网格收集后做一次 NMS，再把坐标从 letterbox 系还原回原图

In [ ]:
from vision_post import (decode_yolo26_pose, unletterbox_boxes,
                         unletterbox_keypoints)

boxes, scores, keypoints = decode_yolo26_pose(outputs, model.output_attrs,
                                              conf_thresh=0.4)
boxes = unletterbox_boxes(boxes, SCALE, PAD_X, PAD_Y, *img.size)
keypoints = unletterbox_keypoints(keypoints, SCALE, PAD_X, PAD_Y)

print(f"检测到 {len(boxes)} 个人:")
for b, s, k in zip(boxes, scores, keypoints):
    vis = (k[:, 2] >= 0.5).sum()
    print(f"  person {s:.2f}  box=({b[0]:.0f},{b[1]:.0f},{b[2]:.0f},{b[3]:.0f})"
          f"  可见关键点 {vis}/17")

## 6. 骨架可视化

COCO 17 关键点按骨架连线（肩-肘-腕、髋-膝-踝等），白点 = 置信度 ≥ 0.5 的关键点。

In [ ]:
from vision_post import draw_pose

result = draw_pose(img, boxes, scores, keypoints)
model.close()
result

## 小结

- YOLO26-pose 的关键点和框在同一个前向里输出（`pos_*` 头 51 通道 = 17 点 × 3），
  不需要单独的 pose 模型二次推理。
- 解码逻辑在 `vision_post.decode_yolo26_pose` / `unletterbox_keypoints`，
  画图在 `vision_post.draw_pose`（`COCO_SKELETON` 定义了 17 条骨架连线）。
- 可以换 n/s 规格对比速度，或换多人照片看骨架串联效果。